# Example: Static forward free-boundary equilibrium calculations (in EAST)

---

Here we will generate an equilibrium (solve for the plasma flux with fixed coil currents) on EAST using the machine geometry from ["PF-Feedforward-EAST: physics-informed imitation learning for feedforward coil-current trajectory design on EAST"](https://github.com/TokFusionAI/PF_Feedforward_EAST).

The coil currents and equilibrium/profile parameters are based approximately on EAST shot 156715 at 5.109 s. Every finite-area active winding pack is represented by a uniform filament grid with a maximum spacing of approximately 3 cm; its original total ampere-turns and circuit wiring are preserved.

Note that the machine description does not include a vacuum vessel!

### Import packages

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

### Create the machine object

In [ ]:
# build machine
from freegsnke import build_machine

tokamak = build_machine.tokamak(
    active_coils_path=f"../machine_configs/EAST/EAST_active_coils.pickle",
    limiter_path=f"../machine_configs/EAST/EAST_limiter.pickle",
    wall_path=f"../machine_configs/EAST/EAST_wall.pickle",
)

In [ ]:
# plot the machine
fig, ax = plt.subplots(figsize=(8, 8), dpi=80)

tokamak.plot(axis=ax, show=False)  # plots the active coils
ax.plot(tokamak.limiter.R, tokamak.limiter.Z, "k-", linewidth=1.2)  # plots the limiter
ax.grid(alpha=0.5)
ax.set_aspect("equal")
ax.set_xlim(0.45, 3.45)
ax.set_ylim(-2.2, 2.2)
ax.set_xlabel(r"Major radius, $R$ [m]")
ax.set_ylabel(r"Height, $Z$ [m]")

### Instantiate an equilibrium

In [ ]:
from freegsnke import equilibrium_update

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,      # provide tokamak object
    Rmin=1.2, Rmax=2.5,   # radial range
    Zmin=-1.3, Zmax=1.3,  # vertical range
    nx=129,               # number of radial grid points (needs to be of the form 2**n + 1)
    ny=129,               # number of vertical grid points (needs to be of the form 2**n + 1)
)

In [ ]:
# initialise the profiles
from freegsnke.jtor_update import GeneralPprimeFFprime

# EFIT profiles from EAST shot 156715 at 5.109 s
psi_n = np.linspace(0.0, 1.0, 129)
pprime_data = 778923.7959794602 * (psi_n - 1.0)
ffprime_data = (
    1.0163643105813263 * psi_n**2
    - 1.9553728814507954 * psi_n
    + 0.9390085708694674
)

profiles = GeneralPprimeFFprime(
    eq=eq,                            # equilibrium object
    Ip=448349.0,                      # plasma current [A]
    fvac=4.6103095348117815,          # fvac = R * B_tor [T m]
    psi_n=psi_n,                      # normalised poloidal flux
    pprime_data=pprime_data,          # pressure derivative profile
    ffprime_data=ffprime_data,        # FF' profile
    Raxis=1.9012964048754795,         # magnetic-axis radius [m]
    Ip_logic=True,
)

In [ ]:
# set the coil currents
# note that PF7 supplies the PF7/PF9 circuit and PF8 supplies the PF8/PF10 circuit
coil_currents = {
    "PF1": 1359.09375,
    "PF2": 1551.1722412109375,
    "PF3": 6525.3671875,
    "PF4": 1948.96630859375,
    "PF5": -4642.3076171875,
    "PF6": 1589.8978271484375,
    "PF7": 9013.955078125,
    "PF8": 8778.06640625,
    "PF11": -9200.0,  # modified from source value to move the plasma away from the outboard wall
    "PF12": -9590.9765625,
    "PF13": 1279.05419921875,
    "PF14": 1260.845703125,
    "IC1": 0.0,
    "IC2": 0.0,
}


eq.tokamak.set_all_coil_currents(
    np.array([coil_currents[name] for name, _ in tokamak.coils])
)

### The forward solve

In [ ]:
# load the static nonlinear solver
from freegsnke import GSstaticsolver
from freegsnke.equilibrium import PsiGuessParabolicJtor

GSStaticSolver = GSstaticsolver.NKGSsolver(eq, seed=42)


# adjust the initial guess to reduce the number of solver iterations
psi_guess = PsiGuessParabolicJtor(
    h = (min(tokamak.limiter.R)+max(tokamak.limiter.R))/2,  # radial centre of ellipse
    k = 0.0,     # vertical centre of ellipse
    a = 0.5,     # approximate radial minor radius of the plasma
    b = 0.5 * 1.8, # approximate vertical minor radius of the plasma
    Ip = profiles.Ip, # plasma current used to scale the guess
    psi_scale = 1.0   # scaling factor for the initial flux guess
)

# calculate the initial plasma flux
plasma_psi = psi_guess.__call__(
    eq=eq,
)

eq.plasma_psi = plasma_psi
# carry out the forward solve with fixed coil currents
GSStaticSolver.forward_solve(
    eq=eq,
    profiles=profiles,
    target_relative_tolerance=1e-7,
    max_solving_iterations=75,
    verbose=True,  # print output
)

### Plot the equilibrium

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), dpi=80)

ax.grid(zorder=0, alpha=0.75)
ax.set_aspect("equal")
eq.tokamak.plot(axis=ax, show=False)  # plots the active coils
eq.plot(axis=ax, show=False)          # plots the equilibrium
ax.plot(tokamak.limiter.R, tokamak.limiter.Z, "k-", linewidth=1.2)  # plots the limiter
ax.set_xlabel(r"Major radius, $R$ [m]")
ax.set_ylabel(r"Height, $Z$ [m]")